# **Validate model**

In [ ]:
import os
import io
import sys
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as preprocess_input_mobilenet
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# ── Edit these paths as needed ──────────────────────────────
model = load_model("best_model.h5")
MODEL_PATH = "best_model.h5"          # Model path
TEST_DIR   = "split_dataset/test"     # Test Path
IMG_SIZE   = (224, 224)               # Image size (MobileNetV2 default)
BATCH_SIZE = 32                       # batch size
SAVE_PLOTS = False                    # True = Save drawings as PNG files

print(f"  Model  : {MODEL_PATH}")
print(f"  Data   : {TEST_DIR}")
print(f"  ImgSize: {IMG_SIZE}")
print(f"  Batch  : {BATCH_SIZE}")

  Model  : best_model.h5
  Data   : split_dataset/test
  ImgSize: (224, 224)
  Batch  : 32


In [ ]:
def preprocess_for_inference(image_input, img_size=224):
    """
    Accepts: path (str), bytes, or PIL Image
    Returns: preprocessed numpy array ready for model inference
    """
    if isinstance(image_input, str):
        img = Image.open(image_input).convert("RGB")
    elif isinstance(image_input, bytes):
        img = Image.open(io.BytesIO(image_input)).convert("RGB")
    else:
        img = image_input.convert("RGB")

    img       = img.resize((img_size, img_size))
    img_array = np.array(img, dtype=np.float32)
    img_array = preprocess_input_mobilenet(img_array)   # scale to [-1, 1]
    return np.expand_dims(img_array, axis=0)             # add batch dimension


def production_predict(image_input, model, classes, top_k=3, img_size=224):
    """
    Production-ready inference with top-k predictions.
    Returns a structured dict with prediction, confidence, tip, and top-k breakdown.
    """
    processed   = preprocess_for_inference(image_input, img_size)
    predictions = model.predict(processed, verbose=0)[0]

    top_indices = predictions.argsort()[-top_k:][::-1]

    result = {
        "prediction": classes[top_indices[0]],
        "confidence": float(predictions[top_indices[0]]),
        "tip"       : RECYCLING_TIPS.get(classes[top_indices[0]], "No tip available."),
        "top_k"     : [
            {"class": classes[i], "probability": float(predictions[i])}
            for i in top_indices
        ],
    }
    return result


def build_test_generator(test_dir, img_size, batch_size):
    """Build test generator without augmentation"""
    datagen = ImageDataGenerator(preprocessing_function=preprocess_input_mobilenet)
    generator = datagen.flow_from_directory(
        test_dir,
        target_size=(img_size, img_size),
        batch_size=batch_size,
        class_mode="categorical",
        shuffle=False,
    )
    return generator


print(" Helper functions defined!")

 Helper functions defined!


In [ ]:
if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(f"Model file not found: {MODEL_PATH}")

model = load_model("best_model.h5")

In [ ]:
if not os.path.exists(TEST_DIR):
    raise FileNotFoundError(f"Test directory not found: {TEST_DIR}")

test_generator = build_test_generator(TEST_DIR, IMG_SIZE[0], BATCH_SIZE)
classes_list   = list(test_generator.class_indices.keys())

print(f"\n Test directory : {TEST_DIR}")
print(f" Classes found  : {classes_list}")
print(f" Total images   : {test_generator.samples}")

Found 761 images belonging to 6 classes.

 Test directory : split_dataset/test
 Classes found  : ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
 Total images   : 761


In [ ]:
test_generator.reset()
test_loss, test_acc = model.evaluate(test_generator, verbose=1)

print(f"\n Test Accuracy : {test_acc * 100:.2f}%")
print(f" Test Loss     : {test_loss:.4f}")

24/24 ━━━━━━━━━━━━━━━━━━━━ 20s 613ms/step - accuracy: 0.9448 - loss: 0.1598

 Test Accuracy : 94.48%
 Test Loss     : 0.1598


In [ ]:
test_generator.reset()
y_pred_probs   = model.predict(test_generator, verbose=1)
y_pred_classes = np.argmax(y_pred_probs, axis=1)
y_true_classes = test_generator.classes

print(f"\n  Total predictions: {len(y_pred_classes)}")

24/24 ━━━━━━━━━━━━━━━━━━━━ 11s 274ms/step

  Total predictions: 761
